In [ ]:
# ============================================
# [0] Install core dependencies (run once per runtime)
# ============================================

!pip install -q pandas numpy spacy tqdm pyarrow fastparquet

# optional extras if not already present
!python -m spacy download en_core_web_sm

# if you plan to visualize or debug quickly
# !pip install -q matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 154.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ============================================
# [A] Mount Drive and define paths
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# ---- BASE PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"   # your project base folder

# ---- INPUTS (recursively load all gov JSON/JSONL) ----
RAW_DIR = f"{BASE}/raw/government_websites"   # contains subfolders: hdb, mas, bca, mnd, sfa, sla

# Property domain resources (SGPropertyDomain)
ENTITYRULER = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
REGEX_JSONL = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"

# ---- OUTPUTS ----
OUTDIR_ROOT = f"{BASE}/preprocess/govenrment_websites"
OUTDIR_BY_AGENCY = f"{OUTDIR_ROOT}/by_agency"

from pathlib import Path
import pandas as pd
import os, re, json, html, unicodedata, hashlib, numpy as np
Path(OUTDIR_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTDIR_BY_AGENCY).mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:


# ============================================
# [B] Auto-merge SGPropertyDomain vocab/*.txt into EntityRuler patterns
# ============================================
VOC_DIR  = f"{BASE}/corpus/SGPropertyDomain/vocab"
EXISTING = ENTITYRULER
MERGED   = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl"

def iter_existing(path):
    out = []
    p = Path(path)
    if not p.exists():
        return out
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except:
                pass
    return out

def phrase_to_token_pattern(phrase: str):
    phrase = re.sub(r"\s+", " ", phrase).strip()
    if not phrase:
        return None
    return [{"LOWER": t.lower()} for t in phrase.split(" ") if t]

def label_from_filename(fname: str):
    stem  = Path(fname).stem
    return re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_").upper() or "DOMAIN"

existing = iter_existing(EXISTING)
voc_path = Path(VOC_DIR)
vocab_patterns = []
if voc_path.exists():
    for txt in sorted(voc_path.glob("*.txt")):
        label = label_from_filename(txt.name)
        for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
            term = raw.strip()
            if not term: continue
            pat = phrase_to_token_pattern(term)
            if pat: vocab_patterns.append({"label": label, "pattern": pat, "id": term})

def lowers_from_pattern(pat):
    if isinstance(pat, str):
        return tuple(p.strip().lower() for p in re.sub(r"\s+"," ",pat).split(" ") if p.strip())
    if isinstance(pat, dict):
        return (str(pat.get("LOWER", pat.get("TEXT",""))).lower(),)
    if isinstance(pat, list):
        outs=[]
        for tok in pat:
            if isinstance(tok, dict):
                outs.append(str(tok.get("LOWER", tok.get("TEXT",""))).lower())
            else:
                outs.append(str(tok).lower())
        return tuple(outs)
    return (str(pat).lower(),)

def pat_key(rec):
    return (rec.get("label",""), lowers_from_pattern(rec.get("pattern","")))

seen=set(); merged=[]
for rec in existing:
    k=pat_key(rec)
    if k in seen: continue
    seen.add(k); merged.append(rec)
for rec in vocab_patterns:
    k=pat_key(rec)
    if k in seen: continue
    seen.add(k); merged.append(rec)

with open(MERGED, "w", encoding="utf-8") as f:
    for rec in merged:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

ENTITYRULER = MERGED
print(f"[EntityRuler] merged → {MERGED} (rules: {len(merged)})")

[EntityRuler] merged → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl (rules: 2158)


In [ ]:
# ============================================
# [C] Helpers (cleaning, loading, regex flags)
# ============================================
def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def deep_clean_text(text: str) -> str:
    """Strong boilerplate remover for HDB/MAS/BCA/MND/SFA/SLA sites."""
    if not isinstance(text, str): return ""
    s = html.unescape(text)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"<[^>]+>", " ", s)
    # URLs/emails
    s = re.sub(r"https?://\S+|www\.\S+|\S+@\S+", " ", s)

    BOILER = [
        # nav/ui
        r"Skip\s*to\s*main\s*content", r"Share\s*this\s*page", r"Print\s*this\s*page",
        r"Back\s*to\s*top", r"(Next|Previous)\s*Page|PrevNext", r"Breadcrumb.*",
        r"Site\s*Map", r"Feedback\s*Form", r"Contact\s*Us",
        r"Subscribe\s*to.*?(updates|newsletter)", r"Follow\s*us\s*on\s*(Facebook|Twitter|LinkedIn|Instagram)",
        r"Terms\s*of\s*Use", r"Privacy\s*Policy", r"Data\s*Protection\s*Statement",
        r"©\s*\d{4}.*?(rights\s*reserved|Government\s*of\s*Singapore)",

        # scaffolds/selectors
        r"Select\s*Year\s*All\d{4}", r"From\s*To\s*Go", r"Search\s*this\s*site",
        r"Table\s*of\s*Contents", r"Related\s*(Pages|Information|Articles)", r"Resources\s*&\s*Downloads",
        r"Download\s*(PDF|Excel|XLSX?|DOCX?)", r"Open\s*in\s*new\s*window", r"Tags\s*:",

        # press-release clutter
        r"About\s*UsPress\s*Releases.*?(Read\s*press\s*release)?",
        r"Read\s*press\s*release", r"Read\s*more", r"Click\s*here\s*to\s*(learn|find)\s*out\s*more",
        r"More\s*details\s*can\s*be\s*found", r"For\s*media\s*queries.*?contact", r"For\s*further\s*enquiries.*?contact",
        r"Photo\s*credit.*", r"Image\s*:\s*(HDB|BCA|SFA|SLA|MAS|MND)|Image\s*credits?:.*",
        r"Annex.*(available|attached).*below", r"Attachment\s*:", r"Press\s*Enquiries", r"Media\s*Release\s*by",
        r"Joint\s*Press\s*Release", r"Source\s*:\s*(HDB|BCA|MAS|MND|SFA|SLA)",
        r"Published\s*Date\s*:\s*\d{1,2}\s*[A-Za-z]{3,}\s*\d{4}",
        r"(Last\s*updated|Updated\s*on).*?\d{4}",

        # disclaimers/footers
        r"This\s*website\s*may\s*contain\s*links\s*to\s*other\s*websites",
        r"The\s*Government\s*shall\s*not\s*be\s*responsible", r"Disclaimer",
        r"All\s*content\s*is\s*available\s*under", r"Creative\s*Commons\s*Attribution",
        r"Cookies\s*policy",

        # crumbs
        r"Home\s*>\s*.*", r"Main\s*Navigation.*", r"Toggle\s*menu",
        r"Quick\s*links", r"FAQ[s]?"
    ]
    for pat in BOILER:
        s = re.sub(pat, " ", s, flags=re.I|re.S)

    s = re.sub(r"([.,!?;:]){2,}", r"\1", s)
    return normalize_ws(s)

def read_json_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".jsonl":
        rows=[]
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for ln in f:
                ln=ln.strip()
                if ln:
                    try: rows.append(json.loads(ln))
                    except: pass
        return pd.json_normalize(rows, max_level=2) if rows else pd.DataFrame()
    else:
        try:
            obj = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
        except Exception:
            return pd.DataFrame()
        if isinstance(obj, list):  return pd.json_normalize(obj, max_level=2)
        if isinstance(obj, dict):  return pd.json_normalize(obj, max_level=2)
        return pd.DataFrame()

def compile_regexes_from_jsonl(path: Path):
    patt=[]
    p=Path(path)
    if not p.exists(): return patt
    with p.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if not ln: continue
            try:
                item=json.loads(ln); pat=item.get("pattern")
                if pat: patt.append({"name": item.get("name","pattern"), "re": re.compile(pat, re.I)})
            except: pass
    return patt

def detect_regex_hits(text: str, compiled):
    hits={}
    for p in compiled:
        try:
            if p["re"].search(text): hits[p["name"]] = True
        except: pass
    return hits

def hash_text(s: str) -> str:
    return hashlib.md5((s or "").strip().lower().encode("utf-8")).hexdigest()

In [ ]:


# ============================================
# [D] Load ALL gov JSON/JSONL, normalize, clean, dedup
# ============================================
raw_paths = sorted(Path(RAW_DIR).rglob("*.json*"))
assert raw_paths, f"No JSON/JSONL found under {RAW_DIR}"

frames=[]
for p in raw_paths:
    d = read_json_any(p)
    if d.empty: continue
    d["source_file"] = p.as_posix()
    for c in ["id","source","text","timestamp","url","language"]:
        if c not in d.columns: d[c] = d.get(c, None)
    # Flatten metadata if present
    if "metadata.title" in d.columns and "title" not in d.columns:           d["title"] = d["metadata.title"]
    if "metadata.agency" in d.columns and "agency" not in d.columns:         d["agency"] = d["metadata.agency"]
    if "metadata.policy_type" in d.columns and "policy_type" not in d.columns: d["policy_type"] = d["metadata.policy_type"]
    if "metadata.location" in d.columns and "location" not in d.columns:     d["location"] = d["metadata.location"]
    frames.append(d)

df = pd.concat(frames, ignore_index=True, sort=False)
print(f"[LOAD] rows={len(df)} files={len(frames)}")

# Clean
df["title"]      = df.get("title","").astype(str)
df["clean_text"] = df.get("text","").astype(str).apply(deep_clean_text)

# Drop very short noise
df = df[(df["clean_text"].str.len() > 80) | (df["title"].str.len() > 10)].copy()

# Early dedup (url/title/text-chunk)
df["_k"] = (
    df.get("url","").astype(str).str.lower() + "||" +
    df["title"].astype(str).str.lower() + "||" +
    df["clean_text"].astype(str).str.lower().str[:400]
)
before=len(df)
df = df.drop_duplicates("_k").drop(columns=["_k"])
print(f"[DEDUP1] removed {before - len(df)}")

# Build body, timestamps
df["body"]    = (df["title"].fillna("") + "\n\n" + df["clean_text"].fillna("")).str.strip()
df["hash"]    = df["body"].apply(hash_text)
df["date"]    = pd.to_datetime(df.get("timestamp"), errors="coerce")
df["quarter"] = df["date"].dt.to_period("Q").astype(str)
df["year"]    = df["date"].dt.year

# Hash-based dedup
before=len(df)
df = df.drop_duplicates("hash").reset_index(drop=True)
print(f"[DEDUP2] removed {before - len(df)} (final rows: {len(df)})")


[LOAD] rows=150 files=23
[DEDUP1] removed 84
[DEDUP2] removed 0 (final rows: 59)


/tmp/ipython-input-1428506798.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["quarter"] = df["date"].dt.to_period("Q").astype(str)


In [ ]:
# ============================================
# [E] Property regex flags + EntityRuler tagging
# ============================================
# Regex flags from REGEX_JSONL
if Path(REGEX_JSONL).exists():
    comp = compile_regexes_from_jsonl(Path(REGEX_JSONL))
    hits = [detect_regex_hits(t, comp) for t in df["clean_text"].astype(str)]
    hdf  = pd.json_normalize(hits)
    if not hdf.empty:
        hdf.columns = [f"rx_{c}" for c in hdf.columns]
        df = pd.concat([df.reset_index(drop=True), hdf.reset_index(drop=True)], axis=1)
        print(f"[REGEX] added {len(hdf.columns)} rx_* columns")

# EntityRuler (domain entities)
try:
    import spacy
    if Path(ENTITYRULER).exists():
        nlp0 = spacy.blank("en")
        ruler = nlp0.add_pipe("entity_ruler")
        ruler.from_disk(ENTITYRULER)
        ents=[]
        for doc in nlp0.pipe(df["clean_text"].astype(str).tolist(), batch_size=64):
            ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
        df["entities"] = ents
    else:
        df["entities"] = [[] for _ in range(len(df))]
except Exception as e:
    print("[WARN] entity_ruler skipped:", e)
    df["entities"] = [[] for _ in range(len(df))]

In [ ]:
# ============================================
# [F] SAVE — per agency + combined
# ============================================
# Per-agency
if "agency" in df.columns:
    for ag in sorted(df["agency"].fillna("UNK").unique()):
        sub = df[df["agency"]==ag].copy()
        sub_out = f"{OUTDIR_BY_AGENCY}/{ag}_gov_enriched_clean.csv"
        sub.to_csv(sub_out, index=False)
        print("saved:", sub_out)

# Combined
COMBINED_CSV = f"{OUTDIR_ROOT}/gov_websites_all_cleaned.csv"
df.to_csv(COMBINED_CSV, index=False)
print("[COMBINED] →", COMBINED_CSV, "| rows:", len(df))

saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/by_agency/BCA_gov_enriched_clean.csv
saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/by_agency/MAS_gov_enriched_clean.csv
saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/by_agency/MND_gov_enriched_clean.csv
saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/by_agency/SFA_gov_enriched_clean.csv
saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/by_agency/SLA_gov_enriched_clean.csv
[COMBINED] → /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/gov_websites_all_cleaned.csv | rows: 59


In [ ]:

# ============================================
# [G] NLP ENRICHMENT (tokens / lemmas / NER / ABSA scaffolding)
# ============================================
RUN_NLP_ENRICHMENT = True
if RUN_NLP_ENRICHMENT:
    try:
        import spacy
        nlp = spacy.load("en_core_web_sm", exclude=[])  # tagger, parser, ner
    except Exception:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy"], check=False)
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
        import spacy
        nlp = spacy.load("en_core_web_sm", exclude=[])

    # Attach domain EntityRuler BEFORE spaCy NER (optional)
    try:
        if Path(ENTITYRULER).exists():
            er = nlp.add_pipe("entity_ruler", before="ner")
            er.from_disk(ENTITYRULER)
    except Exception as e:
        print("[WARN] EntityRuler attach skipped:", e)

    texts = df["clean_text"].astype(str).tolist()
    docs  = list(nlp.pipe(texts, batch_size=64, n_process=2))

    df["tokens"] = [[t.text  for t in d] for d in docs]
    df["lemmas"] = [[t.lemma_ for t in d] for d in docs]
    df["pos"]    = [[t.pos_   for t in d] for d in docs]
    df["deps"]   = [[t.dep_   for t in d] for d in docs]
    df["entities_ner"]      = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in docs]
    df["aspect_candidates"] = [[nc.text for nc in d.noun_chunks] for d in docs]  # ABSA scaffolding

    ENR_DIR = Path(OUTDIR_ROOT) / "nlp_enriched"
    ENR_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(ENR_DIR / "gov_enriched+nlp.parquet", index=False)
    print("Saved:", ENR_DIR / "gov_enriched+nlp.parquet")

    # Sentence table (optional)
    sent_rows=[]; use_date = df["date"].notna().any() if "date" in df.columns else False
    for i, d in enumerate(docs):
        for j, s in enumerate(d.sents):
            row = {"doc_id": i, "sent_id": j, "text": s.text,
                   "tokens": [t.text for t in s],
                   "lemmas": [t.lemma_ for t in s],
                   "pos":    [t.pos_   for t in s],
                   "deps":   [t.dep_   for t in s]}
            if use_date: row["date"] = df.iloc[i]["date"]
            sent_rows.append(row)
    pd.DataFrame(sent_rows).to_parquet(ENR_DIR / "gov_sentences.parquet", index=False)
    print("Saved:", ENR_DIR / "gov_sentences.parquet")

    # Monthly trend if dates exist (posts count only)
    if use_date:
        monthly = (
            df.assign(month=df["date"].dt.to_period("M").astype(str))
              .groupby("month", dropna=True)
              .agg(posts=("clean_text","size"))
              .reset_index()
        )
        monthly.to_csv(ENR_DIR / "monthly_counts.csv", index=False)
        print("Saved:", ENR_DIR / "monthly_counts.csv")
else:
    print("[INFO] RUN_NLP_ENRICHMENT=False → skipping spaCy.")





Saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/nlp_enriched/gov_enriched+nlp.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/nlp_enriched/gov_sentences.parquet


/tmp/ipython-input-2968835580.py:56: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df.assign(month=df["date"].dt.to_period("M").astype(str))


Saved: /content/drive/MyDrive/PropInsight/preprocess/govenrment_websites/nlp_enriched/monthly_counts.csv


In [ ]:
# ============================================
# [H] Quick verification
# ============================================
print("\n=== VERIFICATION ===")
print("Combined CSV exists:", os.path.exists(COMBINED_CSV))
print("By-agency sample:", sorted(os.listdir(OUTDIR_BY_AGENCY))[:6])


=== VERIFICATION ===
Combined CSV exists: True
By-agency sample: ['BCA_gov_enriched_clean.csv', 'MAS_gov_enriched_clean.csv', 'MND_gov_enriched_clean.csv', 'SFA_gov_enriched_clean.csv', 'SLA_gov_enriched_clean.csv']
